In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# A Parser for First Order Logic

## Representing Terms and Formulas

In first order logic, we have to distinguish between *terms* and *formulas*.  Terms are built from variables using function symbols.  A function symbol without arguments is a *constant*.  We represent terms by the discriminated union `Term`:
- `Var "x"` represents the variable $x$,
- `Fun ("F", [t1; t2])` represents the term $F(t_1, t_2)$.  The constant $E$ is represented as `Fun ("E", [])`.

Formulas are represented by the discriminated union `Formula`.  An *atomic formula* $P(t_1, \cdots, t_n)$ is represented as `Atom ("P", [t1; ...; tn])`.  The quantified formulas $\forall x: f$ and $\exists x: f$ are represented as `Forall ("x", f)` and `Exists ("x", f)`.  The remaining constructors are the same as in propositional logic.

In Python, both terms and formulas are represented as nested tuples.  Using two different types has the advantage that the F# compiler can check that, for example, a function symbol is never applied to a formula.

In [2]:
type Term =
    | Var of string                  // variable
    | Fun of string * Term list      // function application, constants have no arguments

type Formula =
    | Verum                          // ⊤
    | Falsum                         // ⊥
    | Atom   of string * Term list   // atomic formula P(t1, ..., tn)
    | Not    of Formula              // ¬
    | And    of Formula * Formula    // ∧
    | Or     of Formula * Formula    // ∨
    | Imp    of Formula * Formula    // →
    | Iff    of Formula * Formula    // ↔
    | Xor    of Formula * Formula    // ⊕
    | Forall of string * Formula     // ∀x: f
    | Exists of string * Formula     // ∃x: f

## Converting Terms and Formulas into Strings

The function `termToString` takes one argument:
- `t` is a term.

It returns a string representing `t`, e.g. `F(x, G(y))`.

In [3]:
let rec termToString (t: Term) : string =
    match t with
    | Var x        -> x
    | Fun (f, ts)  -> f + "(" + String.concat ", " (List.map termToString ts) + ")"

The function `formulaToString` takes one argument:
- `f` is a formula.

It returns a string representing `f`.  Every binary subformula is enclosed in parentheses, so that the structure of the formula is unambiguous.

In [4]:
let rec formulaToString (f: Formula) : string =
    let binary (op: string) (g: Formula) (h: Formula) =
        "(" + formulaToString g + " " + op + " " + formulaToString h + ")"
    match f with
    | Verum          -> "⊤"
    | Falsum         -> "⊥"
    | Atom (p, ts)   -> termToString (Fun (p, ts))
    | Not g          -> "¬" + formulaToString g
    | And (g, h)     -> binary "∧" g h
    | Or  (g, h)     -> binary "∨" g h
    | Imp (g, h)     -> binary "→" g h
    | Iff (g, h)     -> binary "↔" g h
    | Xor (g, h)     -> binary "⊕" g h
    | Forall (x, g)  -> "∀" + x + ":" + formulaToString g
    | Exists (x, g)  -> "∃" + x + ":" + formulaToString g

## The Tokenizer

The function `tokenize` takes one argument:
- `s` is a string that is supposed to represent a formula from first order logic.

It returns the list of tokens of `s`.  The tokens are
- variables, which start with a lower case letter,
- function and predicate symbols, which start with an upper case letter, and
- the logical operators, the quantifiers, and parentheses.

Blanks, tabs, commas, and colons are only used to separate tokens.  They are discarded.

In [5]:
open System.Text.RegularExpressions

let tokenize (s: string) : string list =
    let lexSpec = """([\ ,:\t]+)           |  # blanks, tabs, separators
                     ([a-z][A-Za-z0-9]*)   |  # variables
                     ([A-Z][A-Za-z0-9]*)   |  # function symbols
                     ([⊤⊥∧∨¬→↔⊕∀∃()])        # junctors, quantifiers, '(', ')'
                  """
    [ for m in Regex.Matches(s, lexSpec, RegexOptions.IgnorePatternWhitespace) do
        if not m.Groups[1].Success then
            m.Value ]

In [6]:
tokenize "∀x:∃y:P(F(x), y) → Q(x)"

["∀"; "x"; "∃"; "y"; "P"; "("; "F"; "("; "x"; ")"; "y"; ")"; "→"; "Q"; "("; "x"; ")"]

The function `isVariable` takes one argument:
- `s` is a token.

It checks whether `s` is a variable, i.e. whether `s` starts with a lower case letter.  The function `isFunction` checks whether `s` is a function or predicate symbol, i.e. whether `s` starts with an upper case letter.  Syntactically, we do not distinguish between function symbols and predicate symbols.

In [7]:
let isVariable (s: string) : bool = Regex.IsMatch(s, "^[a-z][A-Za-z0-9]*$")
let isFunction (s: string) : bool = Regex.IsMatch(s, "^[A-Z][A-Za-z0-9]*$")

## A Recursive Descent Parser

The Python version of this parser uses the *shunting yard algorithm*.  As the parser has to produce values of two different types, namely terms and formulas, we use a *recursive descent parser* instead.  For every syntactical category there is one function.  These functions are based on the following grammar, where the operators are listed with increasing precedence:
$$
\begin{array}{lcl}
\textit{formula}   & \rightarrow & \textit{implication} \;[\; \texttt{↔} \; \textit{formula} \;] \\
\textit{implication} & \rightarrow & \textit{exclusive} \;[\; \texttt{→} \; \textit{implication} \;] \\
\textit{exclusive} & \rightarrow & \textit{disjunction} \;\{\; \texttt{⊕} \; \textit{disjunction} \;\} \\
\textit{disjunction} & \rightarrow & \textit{conjunction} \;\{\; \texttt{∨} \; \textit{conjunction} \;\} \\
\textit{conjunction} & \rightarrow & \textit{unary} \;\{\; \texttt{∧} \; \textit{unary} \;\} \\
\textit{unary} & \rightarrow & \texttt{¬}\; \textit{unary} \;\mid\; \texttt{∀}\; \textit{var} \; \textit{unary} \;\mid\; \texttt{∃}\; \textit{var} \; \textit{unary} \;\mid\; \texttt{⊤} \;\mid\; \texttt{⊥} \;\mid\; \texttt{(}\; \textit{formula} \;\texttt{)} \;\mid\; \textit{atom} \\
\textit{atom} & \rightarrow & \textit{Predicate} \;\texttt{(}\; \textit{term} \;\{\; \textit{term} \;\}\; \texttt{)} \;\mid\; \textit{Predicate} \;\texttt{(}\;\texttt{)} \\
\textit{term} & \rightarrow & \textit{var} \;\mid\; \textit{Function} \;\texttt{(}\; \textit{term} \;\{\; \textit{term} \;\}\; \texttt{)} \;\mid\; \textit{Function} \;\texttt{(}\;\texttt{)}
\end{array}
$$
Here, square brackets denote optional parts and curly braces denote parts that can be repeated any number of times.  Hence
- `↔` and `→` are *right associative*, e.g. $p \rightarrow q \rightarrow r$ is parsed as $p \rightarrow (q \rightarrow r)$,
- `⊕`, `∨`, and `∧` are *left associative*,
- quantifiers and negation bind strongest.  For example, $\forall x: P(x) \vee Q(x)$ is parsed as $\bigl(\forall x: P(x)\bigr) \vee Q(x)$.

The colon after a quantified variable and the commas between arguments are discarded by the tokenizer.

**Implementation:** The functions defined below share the mutable variable `tokens`, which contains the tokens that have not yet been processed.  The function `peek` returns the next token without removing it, `next` removes the next token and returns it, and `expect t` removes the next token and checks that it is equal to `t`.  The functions that implement the grammar rules are mutually recursive and are therefore defined using `let rec ... and ...`.

The function `parse` takes one argument:
- `s` is a string representing a formula from first order logic.

It returns the formula represented by `s`.  If `s` is not a well-formed formula, an exception is raised.

In [8]:
let parse (s: string) : Formula =
    let mutable tokens = tokenize s
    let peek () = match tokens with t :: _ -> t | [] -> ""
    let next () =
        match tokens with
        | t :: rest -> tokens <- rest; t
        | []        -> failwithf "syntax error: unexpected end of %s" s
    let expect (t: string) =
        let u = next ()
        if u <> t then failwithf "syntax error: expected %s, found %s in %s" t u s
    /// parse the arguments of a function or predicate symbol, i.e. "(t1, ..., tn)"
    let rec arguments () : Term list =
        expect "("
        let mutable args = []
        while peek () <> ")" do
            args <- args @ [ term () ]
        expect ")"
        args
    and term () : Term =
        let t = next ()
        if isVariable t then Var t
        elif isFunction t then Fun (t, arguments ())
        else failwithf "syntax error: unexpected token %s in term in %s" t s
    let rec formula () : Formula =
        let f = implication ()
        if peek () = "↔" then (next () |> ignore; Iff (f, formula ())) else f
    and implication () : Formula =
        let f = exclusive ()
        if peek () = "→" then (next () |> ignore; Imp (f, implication ())) else f
    and exclusive () : Formula =
        let mutable f = disjunction ()
        while peek () = "⊕" do
            next () |> ignore
            f <- Xor (f, disjunction ())
        f
    and disjunction () : Formula =
        let mutable f = conjunction ()
        while peek () = "∨" do
            next () |> ignore
            f <- Or (f, conjunction ())
        f
    and conjunction () : Formula =
        let mutable f = unary ()
        while peek () = "∧" do
            next () |> ignore
            f <- And (f, unary ())
        f
    and unary () : Formula =
        let t = next ()
        match t with
        | "¬" -> Not (unary ())
        | "∀" -> let x = next () in Forall (x, unary ())
        | "∃" -> let x = next () in Exists (x, unary ())
        | "⊤" -> Verum
        | "⊥" -> Falsum
        | "(" -> let f = formula ()
                 expect ")"
                 f
        | _ when isFunction t -> Atom (t, arguments ())
        | _ -> failwithf "syntax error: unexpected token %s in %s" t s
    let f = formula ()
    if not tokens.IsEmpty then
        failwithf "syntax error: unexpected token %s in %s" (peek ()) s
    f

The function `parseTerm` takes one argument:
- `s` is a string representing a term.

It returns the term represented by `s`.  It is implemented using `parse`: the string `s` is wrapped into an auxiliary predicate symbol `T`.  Then the argument of the resulting atomic formula is extracted.

In [9]:
let parseTerm (s: string) : Term =
    match parse ("T(" + s + ")") with
    | Atom ("T", [ t ]) -> t
    | _                 -> failwithf "syntax error: %s is not a term" s

## Testing

The function `testParser` takes one argument:
- `s` is a string representing a formula.

It parses `s` and prints the resulting formula both as a value of type `Formula` and as a string.

In [10]:
let testParser (s: string) : unit =
    printfn "parsing: \"%s\"" s
    let f = parse s
    print f
    printfn "%s\n" (formulaToString f)

In [11]:
testParser "P(F(x,y),x)"
testParser "P(F(x),G(z))"
testParser "∀x:∃y:P(x,y)"
testParser "∀x:∃y:P(x,y)→∃y:∀x:P(x,y)"
testParser "¬∀x:(Red(x) → Happy(x))"
testParser "∀x:∀y:(¬P(F(x),y)) ∨ ∀u:∀v:(¬P(u,G(v)))"

parsing: "

P(F(x,y),x)

"

Atom ("P", [Fun ("F", [Var "x"; Var "y"]); Var "x"])

P(F(x, y), x)

...

parsing: "

∀x:∀y:(¬P(F(x),y)) ∨ ∀u:∀v:(¬P(u,G(v)))

"

Or (Forall ("x", Forall ("y", Not (Atom ("P", [Fun ("F", [Var "x"]); Var "y"])))), Forall ("u", Forall ("v", Not (Atom ("P", [Var "u"; Fun ("G", [Var "v"])])))))

(∀x:∀y:¬P(F(x), y) ∨ ∀u:∀v:¬P(u, G(v)))

In [12]:
parseTerm "G(F(x,y),x)"

Fun ("G", [Fun ("F", [Var "x"; Var "y"]); Var "x"])